In [ ]:
# Colab: 安裝本 notebook 需要的套件
!pip install -q pypdf rank-bm25 sentence-transformers faiss-cpu langchain requests


In [ ]:
import os
import re
import glob
import requests
import numpy as np
from typing import List, Tuple, Dict, Any
from dataclasses import dataclass

from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


# =========================
# Colab / Ollama Cloud 設定
# =========================
# 在 Colab 建議用左側 Secrets 設定 OLLAMA_API_KEY；或在執行前用 os.environ 設定。
# 直接呼叫 Ollama Cloud API 時，模型名稱通常使用 gpt-oss:120b。
# CLI 的 cloud model 名稱 gpt-oss:120b-cloud 主要用於本機 ollama run / pull。
try:
    from google.colab import userdata
    COLAB_OLLAMA_API_KEY = userdata.get("OLLAMA_API_KEY")
except Exception:
    COLAB_OLLAMA_API_KEY = None

OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY") or COLAB_OLLAMA_API_KEY or ""
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "https://ollama.com").rstrip("/")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gpt-oss:120b")

DATA_DIR = "data"
# BAAI/bge-m3 是 multilingual embedding model，適合中英混合與繁體中文 RAG。
EMBED_MODEL_NAME = "BAAI/bge-m3"
TOP_K = 5
MULTI_QUERY_COUNT = 3
REQUEST_TIMEOUT = 120

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", "。", "！", "？", "；", "，", " ", ""],
)

embedder = SentenceTransformer(EMBED_MODEL_NAME)


# =========================
# Ollama Cloud Chat API
# =========================
def require_ollama_api_key() -> None:
    if not OLLAMA_API_KEY:
        raise ValueError(
            "找不到 OLLAMA_API_KEY。請在 Colab Secrets 新增 OLLAMA_API_KEY，"
            "或先執行：os.environ['OLLAMA_API_KEY'] = '你的 API key'"
        )


def ollama_chat(messages: List[Dict[str, str]], model: str = OLLAMA_MODEL, temperature: float = 0.0) -> str:
    """Call Ollama Cloud /api/chat and return assistant text."""
    require_ollama_api_key()

    payload: Dict[str, Any] = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature},
    }
    headers = {
        "Authorization": f"Bearer {OLLAMA_API_KEY}",
        "Content-Type": "application/json",
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        headers=headers,
        json=payload,
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    data = response.json()
    return data.get("message", {}).get("content", "").strip()


# =========================
# PDF 讀取與切塊
# =========================
def load_pdf_text(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    pages = []
    for page_no, page in enumerate(reader.pages, start=1):
        try:
            pages.append(page.extract_text() or "")
        except Exception as exc:
            print(f"Warning: {pdf_path} page {page_no} 讀取失敗：{exc}")
            pages.append("")
    return "\n".join(pages).strip()


def build_documents(data_dir: str) -> List[Document]:
    pdf_files = sorted(glob.glob(os.path.join(data_dir, "**/*.pdf"), recursive=True))
    if not pdf_files:
        raise FileNotFoundError(f"找不到 PDF。請把 PDF 放到 Colab 的 {data_dir}/ 資料夾。")

    documents: List[Document] = []
    for pdf_path in pdf_files:
        full_text = load_pdf_text(pdf_path)
        if not full_text:
            print(f"Warning: {pdf_path} 沒有抽取到文字，已略過。")
            continue

        chunks = text_splitter.split_text(full_text)
        for i, chunk in enumerate(chunks):
            documents.append(
                Document(
                    page_content=chunk,
                    metadata={"source": pdf_path, "chunk_id": i},
                )
            )

    if not documents:
        raise ValueError("PDF 已找到，但沒有可用文字 chunk。若是掃描檔，請先 OCR。")
    return documents


# =========================
# Dense Index: FAISS
# =========================
@dataclass
class FaissStore:
    index: faiss.IndexFlatIP
    embeddings: np.ndarray
    documents: List[Document]


def embed_texts(texts: List[str], show_progress_bar: bool = False) -> np.ndarray:
    vectors = embedder.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=show_progress_bar,
    )
    return np.asarray(vectors, dtype="float32")


def build_faiss_store(documents: List[Document]) -> FaissStore:
    texts = [d.page_content for d in documents]
    vectors = embed_texts(texts, show_progress_bar=True)
    dim = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(vectors)
    return FaissStore(index=index, embeddings=vectors, documents=documents)


def faiss_search(store: FaissStore, query: str, top_k: int = TOP_K) -> List[Tuple[Document, float]]:
    k = min(top_k, len(store.documents))
    q = embed_texts([query])
    scores, idxs = store.index.search(q, k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1:
            continue
        results.append((store.documents[idx], float(score)))
    return results


# =========================
# Sparse Index: BM25
# =========================
def tokenize(text: str) -> List[str]:
    """Simple mixed Chinese/English tokenizer for BM25."""
    return re.findall(r"[a-zA-Z0-9_]+|[\u4e00-\u9fff]", text.lower())


def build_bm25(documents: List[Document]) -> BM25Okapi:
    corpus_tokens = [tokenize(d.page_content) for d in documents]
    return BM25Okapi(corpus_tokens)


def bm25_search(bm25: BM25Okapi, documents: List[Document], query: str, top_k: int = TOP_K) -> List[Tuple[Document, float]]:
    scores = bm25.get_scores(tokenize(query))
    k = min(top_k, len(documents))
    top_idx = np.argsort(scores)[::-1][:k]
    return [(documents[i], float(scores[i])) for i in top_idx]


# =========================
# Multi-Query RAG
# =========================
def generate_multi_queries(question: str) -> List[str]:
    system_prompt = (
        "你是一個查詢重寫助手。請根據使用者問題產生不同角度的檢索查詢。"
        "輸出每行一個查詢，不要解釋。"
    )
    resp = ollama_chat(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"問題：{question}\n請產生 {MULTI_QUERY_COUNT} 個查詢。"},
        ],
        temperature=0.2,
    )

    queries = []
    for line in resp.splitlines():
        q = re.sub(r"^\s*[-*\d.、)）]+\s*", "", line).strip()
        if q and q not in queries:
            queries.append(q)

    if question not in queries:
        queries.insert(0, question)
    return queries[:MULTI_QUERY_COUNT]


# =========================
# Hybrid Retrieval
# =========================
def document_key(doc: Document) -> Tuple[str, int]:
    return (doc.metadata.get("source", ""), int(doc.metadata.get("chunk_id", -1)))


def dedupe_documents(items: List[Tuple[Document, float]]) -> List[Tuple[Document, float]]:
    seen = set()
    out = []
    for doc, score in items:
        key = document_key(doc)
        if key in seen:
            continue
        seen.add(key)
        out.append((doc, score))
    return out


def hybrid_search(
    faiss_store: FaissStore,
    bm25: BM25Okapi,
    documents: List[Document],
    query: str,
    top_k: int = TOP_K,
) -> List[Tuple[Document, float]]:
    dense_hits = faiss_search(faiss_store, query, top_k=top_k)
    sparse_hits = bm25_search(bm25, documents, query, top_k=top_k)
    combined: Dict[Tuple[str, int], Dict[str, Any]] = {}

    # Reciprocal-rank style fusion. BM25 raw scores and dense scores are on different scales,
    # so rank-based scores are more stable for a teaching demo.
    for rank, (doc, _) in enumerate(dense_hits, start=1):
        key = document_key(doc)
        combined.setdefault(key, {"doc": doc, "score": 0.0})
        combined[key]["score"] += 0.65 / rank

    for rank, (doc, _) in enumerate(sparse_hits, start=1):
        key = document_key(doc)
        combined.setdefault(key, {"doc": doc, "score": 0.0})
        combined[key]["score"] += 0.35 / rank

    ranked = sorted(combined.values(), key=lambda x: x["score"], reverse=True)
    return [(item["doc"], float(item["score"])) for item in ranked[:top_k]]


def multi_query_hybrid_retrieve(
    faiss_store: FaissStore,
    bm25: BM25Okapi,
    documents: List[Document],
    question: str,
    top_k: int = TOP_K,
) -> List[Tuple[Document, float]]:
    queries = generate_multi_queries(question)
    print("Generated queries:", queries)

    all_hits: List[Tuple[Document, float]] = []
    for q in queries:
        all_hits.extend(hybrid_search(faiss_store, bm25, documents, q, top_k=top_k))

    merged: Dict[Tuple[str, int], Dict[str, Any]] = {}
    for doc, score in dedupe_documents(all_hits):
        key = document_key(doc)
        merged.setdefault(key, {"doc": doc, "score": 0.0})
        merged[key]["score"] += score

    ranked = sorted(merged.values(), key=lambda x: x["score"], reverse=True)
    return [(item["doc"], float(item["score"])) for item in ranked[:top_k]]


# =========================
# Answer Generation
# =========================
def build_context(retrieved: List[Tuple[Document, float]]) -> str:
    blocks = []
    for i, (doc, score) in enumerate(retrieved, start=1):
        src = doc.metadata.get("source", "unknown")
        cid = doc.metadata.get("chunk_id", -1)
        blocks.append(f"[{i}] source={src}, chunk={cid}, score={score:.4f}\n{doc.page_content}")
    return "\n\n".join(blocks)


def ask_rag(question: str, faiss_store: FaissStore, bm25: BM25Okapi, documents: List[Document]) -> str:
    retrieved = multi_query_hybrid_retrieve(faiss_store, bm25, documents, question, top_k=TOP_K)
    context = build_context(retrieved)
    messages = [
        {
            "role": "system",
            "content": "你是專業的文件問答助手。只能根據提供的上下文回答；若上下文不足，請明確說不知道。",
        },
        {
            "role": "user",
            "content": f"問題：{question}\n\n上下文：\n{context}\n\n請用繁體中文回答，並列出你使用到的重點依據。",
        },
    ]
    return ollama_chat(messages, temperature=0.0)


# =========================
# 主流程
# =========================
print(f"Embedding model: {EMBED_MODEL_NAME}")
print(f"Ollama host: {OLLAMA_HOST}")
print(f"Ollama model: {OLLAMA_MODEL}")
require_ollama_api_key()

print("Loading PDFs...")
docs = build_documents(DATA_DIR)
print(f"Loaded chunks: {len(docs)}")

print("Building FAISS index...")
faiss_store = build_faiss_store(docs)

print("Building BM25 index...")
bm25 = build_bm25(docs)

print("\nRAG ready. Type 'exit' to stop.\n")

while True:
    question = input("請輸入問題：").strip()
    if question.lower() in {"exit", "quit"}:
        break
    if not question:
        continue

    answer = ask_rag(question, faiss_store, bm25, docs)
    print("\n=== 回答 ===")
    print(answer)
    print("\n" + "=" * 40 + "\n")
